# 07 — Layer analysis: leaving vs giving-the-answer in the baseline model

*Auto-executes after Q0's cache lands. The question behind it (user, 09-01):
the every-round reminder prevents leaks only by driving universal exits — can
the residual stream give us something better than a prompt: a direction whose
manipulation keeps the tutor TUTORING (neither leak nor bail)? First step:
where in the stack, and at which token position, are the two consequential
actions decodable — and are they one axis or two?*

Target-state frame (user's log): three classes — leak (failure), abandon
(failure of a different kind), genuinely tutor (target). A useful intervention
direction is one that moves probability OUT of leak without dumping it into
abandon. Exploratory relative to PREREG (Amendment 3's primary test is
untouched); the leak-vs-leave contrast here is the design study for Arm 2.


In [ ]:
import glob, json as js, warnings
import numpy as np
import matplotlib.pyplot as plt
warnings.filterwarnings("ignore")

CACHE = "../microscope/cache/qwen35-9b-v1"
trials = []
for f in sorted(glob.glob(CACHE + "/*.npz")):
    mf = f.replace(".npz", ".json")
    try:
        z = np.load(f); meta = js.load(open(mf))
    except Exception as e:
        print("skip", f, e); continue
    trials.append({"z": z, "m": meta})
print(f"cached trials: {len(trials)}")
if trials:
    z0 = trials[0]["z"]
    HAS_USER = "resid_user" in z0
    n_layers = z0["resid"].shape[1]
    print("keys:", list(z0.keys()), "| layers:", n_layers,
          "| pre-decision states:", HAS_USER)
else:
    print("NO CACHE — run after Q0.")


## Dataset: two contrasts, two token positions

- **C-A (drift):** supportive baseline, pre-event rounds — will-leak vs
  never-leak (the primary-test contrast, previewed in 04).
- **C-B (action type — this notebook's focus):** across personas, pre-event
  rounds of trials that END in a leak vs trials that END in an exit. Round- and
  persona-mix reported; persona-generalization checked explicitly since action
  correlates with persona (aggressor→leak r1 excluded as always).

Positions: `resid` (end of tutor reply — post-turn state) and `resid_user`
(end of student turn — pre-decision state; the sharper reading for "what will
the model do next").


In [ ]:
def rows_for(trials, position):
    key, rkey = (("resid", "resid_rounds") if position == "assistant"
                 else ("resid_user", "user_rounds"))
    out = []
    for t in trials:
        m, z = t["m"], t["z"]
        if key not in z: continue
        leaked = str(m["outcome"]).startswith("leak")
        left = m["outcome"] == "left"
        ev = m["leak_round"] if leaked else m["leave_round"]
        if ev is None or (leaked and ev == 1): continue   # r1 leaks: no window
        R = np.asarray(z[rkey])
        for i, rnd in enumerate(R):
            # pre-event only; for user position, round rnd's student turn
            # precedes round rnd's reply -> allow rnd <= ev-1 (assistant) or
            # rnd < ev (user: the ev-round student turn is still pre-decision
            # ... but its reply IS the event -> exclude to stay strictly
            # pre-event across both positions)
            if rnd >= ev: continue
            out.append({"trial": m["trial"], "persona": m["persona"],
                        "round": int(rnd), "x": z[key][i].astype(np.float32),
                        "leaked": leaked, "left": left})
    return out

if trials:
    data = {p: rows_for(trials, p) for p in
            (["assistant", "user"] if HAS_USER else ["assistant"])}
    for p, rows in data.items():
        n_lk = sum(r["leaked"] for r in rows); n_lv = sum(r["left"] for r in rows)
        print(f"{p:9s}: {len(rows)} pre-event rows | from leak-trials {n_lk}, "
              f"exit-trials {n_lv}")


## Per-layer decodability — C-B action type, both positions

GroupKFold by trial; shuffled-label band; AUROC per layer.


In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import GroupKFold, cross_val_predict

def per_layer_auc(rows, n_layers, seed=0):
    X = np.stack([r["x"] for r in rows]);
    y = np.array([r["leaked"] for r in rows])
    g = np.array([r["trial"] for r in rows])
    rng = np.random.default_rng(seed)
    def auc_of(Xl, yy):
        cv = GroupKFold(min(5, len(np.unique(g))))
        pipe = make_pipeline(StandardScaler(),
                             LogisticRegression(max_iter=2000))
        s = cross_val_predict(pipe, Xl, yy, cv=cv, groups=g,
                              method="predict_proba")[:, 1]
        pos, neg = s[yy], s[~yy]
        return ((pos[:, None] > neg[None, :]).sum()
                + 0.5 * (pos[:, None] == neg[None, :]).sum())             / (len(pos) * len(neg))
    a = [auc_of(X[:, L], y) for L in range(n_layers)]
    ysh = rng.permutation(y)
    ash = [auc_of(X[:, L], ysh) for L in range(0, n_layers, 4)]
    return np.array(a), np.array(ash)

if trials:
    fig, axes = plt.subplots(1, len(data), figsize=(6 * len(data), 3),
                             sharey=True)
    axes = np.atleast_1d(axes)
    results = {}
    for ax, (p, rows) in zip(axes, data.items()):
        keep = [r for r in rows if r["leaked"] or r["left"]]
        a, ash = per_layer_auc(keep, n_layers)
        results[p] = a
        ax.plot(a, label="leak-vs-exit probe")
        ax.axhspan(min(0.5, ash.min()), max(0.5, ash.max()),
                   color="gray", alpha=0.2, label="shuffled range")
        best = int(np.argmax(a))
        ax.set_title(f"{p} position (best L{best}={a[best]:.2f})", fontsize=10)
        ax.set_xlabel("layer"); ax.axhline(0.5, ls=":", c="k", lw=0.5)
    axes[0].set_ylabel("AUROC (grouped CV)"); axes[0].legend(fontsize=8)
    plt.tight_layout(); plt.show()


### Persona-generalization check (action ~ persona confound)

Train leak-vs-exit on supportive only, test on neutral (and vice versa), at the
best layer per position. A probe that survives the swap decodes the ACTION
tendency, not the script.


In [ ]:
if trials:
    from sklearn.metrics import roc_auc_score
    for p, rows in data.items():
        keep = [r for r in rows if r["leaked"] or r["left"]]
        best = int(np.argmax(results[p]))
        for tr_p, te_p in (("supportive", "neutral"), ("neutral", "supportive")):
            tr = [r for r in keep if r["persona"] == tr_p]
            te = [r for r in keep if r["persona"] == te_p]
            ytr = np.array([r["leaked"] for r in tr])
            yte = np.array([r["leaked"] for r in te])
            if ytr.sum() < 3 or (~ytr).sum() < 3 or yte.sum() < 2 or (~yte).sum() < 2:
                print(f"{p} L{best} {tr_p}->{te_p}: insufficient class balance"); continue
            pipe = make_pipeline(StandardScaler(),
                                 LogisticRegression(max_iter=2000))
            pipe.fit(np.stack([r["x"][best] for r in tr]), ytr)
            s = pipe.predict_proba(np.stack([r["x"][best] for r in te]))[:, 1]
            print(f"{p} L{best} {tr_p}->{te_p}: AUROC {roc_auc_score(yte, s):.2f} "
                  f"(n={len(te)})")


## Direction geometry — one axis or two?

Diff-in-means for leak-vs-exit per layer; adjacent-layer cosine (is one
feature carried through the stack?); and cosine between the ACTION axis and
the DRIFT axis (C-A's will-leak-vs-never direction) at the best layer. If the
two axes are near-orthogonal, "stop the leak" and "stop the exit" are
separately steerable — the intervention story better than a prompt.


In [ ]:
if trials:
    p = "user" if HAS_USER else "assistant"
    rows = [r for r in data[p] if r["leaked"] or r["left"]]
    X = np.stack([r["x"] for r in rows]); y = np.array([r["leaked"] for r in rows])
    dm = np.zeros((n_layers, X.shape[-1]), np.float32)
    for L in range(n_layers):
        d = X[y, L].mean(0) - X[~y, L].mean(0); dm[L] = d / np.linalg.norm(d)
    adj = [float(dm[L] @ dm[L + 1]) for L in range(n_layers - 1)]
    fig, ax = plt.subplots(figsize=(7, 2.4))
    ax.plot(adj); ax.set_xlabel("layer"); ax.set_ylabel("cos(L, L+1)")
    ax.set_title("action-axis stability through the stack"); plt.tight_layout(); plt.show()

    sup = [r for r in data[p] if r["persona"] == "supportive"]
    ys = np.array([r["leaked"] for r in sup])
    if ys.sum() >= 3 and (~ys).sum() >= 3:
        Xs = np.stack([r["x"] for r in sup])
        best = int(np.argmax(results[p]))
        d_drift = Xs[ys, best].mean(0) - Xs[~ys, best].mean(0)
        d_drift /= np.linalg.norm(d_drift)
        print(f"cos(action axis, drift axis) at L{best}: "
              f"{float(dm[best] @ d_drift):+.2f}")


## Discussion — what licenses an intervention better than the prompt

Read tonight's outputs against these pre-written patterns:
1. **Action decodable pre-event at mid layers + survives persona swap** → the
   state knows WHICH capitulation is coming; an Arm-2 target exists.
2. **Action axis ≠ drift axis (low cosine)** → two knobs: ablating drift may
   reduce leaks WITHOUT forcing exits — precisely what the reminder cannot do.
   High cosine → one capitulation axis, and any intervention must be dosed to
   land in "keep tutoring," not overshoot into exit.
3. **`user` position beats `assistant`** → the decision is present after
   reading, before speaking — steer at the student-turn boundary.
4. Nothing decodable → interventions are premature; the reminder's
  substitution effect stands as the honest headline.
